# The Price is Right

Today we'll build another piece of the puzzle: a ScanningAgent that looks for promising deals by subscribing to RSS feeds.

In [1]:
# imports

import os
import json
from dotenv import load_dotenv
from openai import OpenAI
from agents.deals import ScrapedDeal, DealSelection

In [2]:
# Initialize and constants

load_dotenv(override=True)
os.environ['OPENAI_API_KEY'] = os.getenv('OPENAI_API_KEY', 'your-key-if-not-using-env')
MODEL = 'gpt-4o-mini'
openai = OpenAI()

In [3]:
deals = ScrapedDeal.fetch(show_progress=True)

100%|██████████| 5/5 [03:05<00:00, 37.19s/it]


In [4]:
len(deals)

50

In [5]:
deals[44].describe()

"Title: Refurb Dyson TP7A Purifier Cool Autoreact Fan for $190 + free shipping\nDetails: That's the best price we've seen in any condition and other sellers have it for $300 or more. It includes a 2-year Allstate warranty. Buy Now at eBay\nFeatures: 6 modes More than 215 sq ft of coverage Model: 419865-01\nURL: https://www.dealnews.com/products/Dyson/Dyson-TP7-A-Purifier-Cool-Autoreact-Fan/491984.html?iref=rss-c196"

In [6]:
system_prompt = """You identify and summarize the 5 most detailed deals from a list, by selecting deals that have the most detailed, high quality description and the most clear price.
Respond strictly in JSON with no explanation, using this format. You should provide the price as a number derived from the description. If the price of a deal isn't clear, do not include that deal in your response.
Most important is that you respond with the 5 deals that have the most detailed product description with price. It's not important to mention the terms of the deal; most important is a thorough description of the product.
Be careful with products that are described as "$XXX off" or "reduced by $XXX" - this isn't the actual price of the product. Only respond with products when you are highly confident about the price. 

{"deals": [
    {
        "product_description": "Your clearly expressed summary of the product in 4-5 sentences. Details of the item are much more important than why it's a good deal. Avoid mentioning discounts and coupons; focus on the item itself. There should be a paragpraph of text for each item you choose.",
        "price": 99.99,
        "url": "the url as provided"
    },
    ...
]}"""

In [7]:
user_prompt = """Respond with the most promising 5 deals from this list, selecting those which have the most detailed, high quality product description and a clear price.
Respond strictly in JSON, and only JSON. You should rephrase the description to be a summary of the product itself, not the terms of the deal.
Remember to respond with a paragraph of text in the product_description field for each of the 5 items that you select.
Be careful with products that are described as "$XXX off" or "reduced by $XXX" - this isn't the actual price of the product. Only respond with products when you are highly confident about the price. 

Deals:

"""
user_prompt += '\n\n'.join([deal.describe() for deal in deals])

In [8]:
print(user_prompt[:2000])

Respond with the most promising 5 deals from this list, selecting those which have the most detailed, high quality product description and a clear price.
Respond strictly in JSON, and only JSON. You should rephrase the description to be a summary of the product itself, not the terms of the deal.
Remember to respond with a paragraph of text in the product_description field for each of the 5 items that you select.
Be careful with products that are described as "$XXX off" or "reduced by $XXX" - this isn't the actual price of the product. Only respond with products when you are highly confident about the price. 

Deals:

Title: 2-Year Mint Unlimited Plan w/ New Phone From $30/mo + free shipping
Details: New customers can buy a new phone, bundled with a 2-year Mint Unlimited plan, with prices starting from $30 per month — available phones include the iPhone 16 Pro Max, Samsung Galaxy S25 Ultra, and Google Pixel 9. It requires port-in, and upfront payment for the device and 24 months of serv

In [10]:
def get_recommendations():
    completion = openai.beta.chat.completions.parse(
        model="gpt-4o-mini",
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_prompt}
      ],
        response_format=DealSelection
    )
    result = completion.choices[0].message.parsed
    return result

In [11]:
result = get_recommendations()

In [12]:
len(result.deals)

5

In [13]:
result.deals[1]

Deal(product_description="The H&A AC60 Hypercardioid Studio Microphone boasts a frequency range of 20 Hz to 20 kHz, making it ideal for capturing high-quality audio. Its noise-reducing hypercardioid pattern ensures that unwanted ambient noise is minimized, letting your voice shine through. The durable metal housing ensures longevity, while custom sound switches provide flexibility for various recording environments. Now available for just $50 with free shipping, it's a great addition to any audio setup.", price=50.0, url='https://www.dealnews.com/H-A-AC60-Hypercardioid-Studio-Microphone-for-50-free-shipping/21754968.html?iref=rss-c142')

In [14]:
from agents.scanner_agent import ScannerAgent

In [15]:
agent = ScannerAgent()
result = agent.scan()

In [16]:
result

DealSelection(deals=[Deal(product_description='The Refurb Unlocked Samsung Galaxy S25 is a powerful smartphone featuring a 128GB storage capacity. It boasts a stunning display perfect for media consumption and comes with a 1-year Allstate warranty for peace of mind. The phone is designed to support high-performance applications and gaming, ensuring smooth operation. With its sleek design and advanced camera system, it is an excellent choice for those looking for quality in a smartphone.', price=496.0, url='https://www.dealnews.com/products/Samsung/Unlocked-Samsung-Galaxy-S25-128-GB-Android-Smartphone/491989.html?iref=rss-c142'), Deal(product_description='The H&A AC60 Hypercardioid Studio Microphone is engineered for high-quality audio capture, making it ideal for professional studios or home recording setups. With a frequency range of 20 Hz to 20 kHz and a noise-reducing hypercardioid pattern, this microphone ensures clarity and detail in sound. Its durable metal housing and custom sou